# 游戏评论洞察 Agent · 完整流程演示

本 notebook 展示一条评论如何从**原始文本**走到**可决策的洞察图表**：

> 输入评论 → LLM 情感分类 → LLM 议题分类 → 群体差异检验 → matplotlib 可视化

**混合模式**：第 1 步在单条样例上做真实 LLM 调用演示；第 2 步起加载预跑好的全量结果
（`agent_full_results.csv`），因此**没有 API Key 也能从头跑到出图**。


## 0 · 环境准备


In [ ]:
import sys, os
from pathlib import Path

# 无论从哪个目录启动，都定位到项目根
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 关键：让 matplotlib 正常显示中文（否则是豆腐块）
plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei"]
plt.rcParams["axes.unicode_minus"] = False

SENT_COLOR = {"positive": "#4CAF50", "neutral": "#FFC107", "negative": "#F44336"}
print("工作目录:", os.getcwd())


## 1 · 单条评论真实调用演示

拿一条**混合情感**的评论（有夸有骂、结尾还想继续玩），看 Agent 怎么判。
若本机没装 `openai` 或没配 `DEEPSEEK_API_KEY`，会自动降级为预置结果，不影响后续。


In [ ]:
sample = "美术和世界观是真顶，但抽卡保底太贵了，新手引导也有点劝退，不过总体还是想继续玩下去"

try:
    from agent import classify, classify_topics, LLMClient
    client = LLMClient()
    sent = classify(sample, client)
    topic = classify_topics(sample, client)
    print("[真实 LLM 调用]")
except Exception as e:
    print(f"[无 API/依赖，降级为预置结果] {e}")
    sent = {"sentiment": "neutral", "confidence": 0.65}
    # 议题标签取自 6 类预定义体系：玩法/商业化/技术/角色/运营/其他
    topic = {"topics": ["角色", "商业化", "玩法"], "primary_topic": "商业化",
             "confidence": 0.7, "reason": "夸美术/角色但主要吐槽抽卡定价与新手引导，结尾表达继续游玩意愿"}

print("\n评论：", sample)
print("情感：", sent)
print("议题：", topic)


## 2 · 加载全量预跑结果

2000+ 条评论的情感 + 议题标注已批量跑好并落盘，这里直接加载做聚合分析。


In [ ]:
df = pd.read_csv("agent_full_results.csv")
df["ver"] = df["version"].astype(str).str.extract(r"(\d\.\d)")
print("总评论数:", len(df))
# content_preview 在两条流水线各带一份（_s 情感 / _t 议题），取情感侧做展示
df[["content_preview_s", "sentiment", "primary_topic", "topics", "version"]].head()


## 3 · 整体情感分布


In [ ]:
order = ["positive", "neutral", "negative"]
counts = df["sentiment"].value_counts().reindex(order).fillna(0)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(["正面", "中性", "负面"], counts.values,
       color=[SENT_COLOR[s] for s in order])
for i, v in enumerate(counts.values):
    ax.text(i, v + 8, f"{int(v)}\n({v/len(df):.0%})", ha="center")
ax.set_title("整体情感分布")
ax.set_ylabel("评论数")
plt.tight_layout()
plt.show()


## 4 · 各议题负面率（找短板）

按主议题分组，看哪个议题被骂得最狠——这是给策划的优先级依据。


In [ ]:
g = df.groupby("primary_topic")["sentiment"]
stat = pd.DataFrame({
    "评论数": g.size(),
    "负面率": g.apply(lambda s: (s == "negative").mean()),
}).sort_values("负面率", ascending=False)

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.barh(stat.index[::-1], stat["负面率"].values[::-1], color="#F44336")
for i, (rate, n) in enumerate(zip(stat["负面率"].values[::-1], stat["评论数"].values[::-1])):
    ax.text(rate + 0.01, i, f"{rate:.0%}  (n={n})", va="center")
ax.set_xlim(0, 1)
ax.set_title("各议题负面率")
ax.set_xlabel("负面评论占比")
plt.tight_layout()
plt.show()
stat.round(3)


## 5 · 版本间差异显著性检验

1.0 公测 vs 1.3，各议题情感是否**统计显著**改善？用卡方 + Cramer's V 效应量判断。
（`p<0.05` 差异非随机；`Cramer's V` 衡量差异大小，不被大样本刷爆）


In [ ]:
from eval.aggregators import load_results, batch_compare

data = load_results()
cmp = batch_compare(data, ["1.0"], ["1.3"])
cmp


In [ ]:
# 按效应量排序可视化：谁回暖最多
plot_df = cmp.dropna(subset=["cramers_v"]).sort_values("cramers_v")

fig, ax = plt.subplots(figsize=(7, 4))
colors = ["#4CAF50" if s else "#BDBDBD" for s in plot_df["significant"]]
ax.barh(plot_df["topic"], plot_df["cramers_v"], color=colors)
for i, (v, sig) in enumerate(zip(plot_df["cramers_v"], plot_df["significant"])):
    ax.text(v + 0.005, i, f"{v:.2f}{' *' if sig else ''}", va="center")
ax.axvline(0.1, ls="--", c="gray", lw=1)
ax.set_title("1.0 → 1.3 各议题改善效应量 (Cramer's V)")
ax.set_xlabel("效应量（虚线=0.1 弱效应门槛，* = p<0.05 显著）")
plt.tight_layout()
plt.show()


## 6 · 结论

- 1.0→1.3 **全议题负面率显著下降**，口碑全面回暖（每项 p<0.05）
- **角色 / 其他** 改善效应最强（V>0.3）；**商业化 / 技术** 虽显著但效应最弱，仍是相对短板
- ⚠️ 版本对比存在**幸存者偏差**：不同时期是不同评论者，负面率下降部分源自不满玩家流失，
  结论应表述为「当前活跃评论人群口碑优于公测期」而非「游戏让所有人更满意」
